# M1 Class Binary Boosting Gate 검증

## tl;dr

`normal vs fault`, `normal vs task`, `normal vs activity`를 각각 분리하고 Logistic 기준선 대비 LightGBM/XGBoost 개선 여부를 검증한다.

## Context & Methods

- 19번 one-vs-normal dataset 정책을 입력 기준으로 사용한다.
- feature는 compact13, expanded154를 비교한다.
- model은 Dummy, Logistic, LightGBM, XGBoost를 비교한다.
- validation은 substation_id 기준 StratifiedGroupKFold를 유지한다.

In [1]:

from __future__ import annotations

from pathlib import Path
import subprocess
import warnings

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / "07_데이터산출물"
SOURCE_DIR = PROJECT_ROOT / "05_데이터셋" / "PreDist"
ZIP_PATH = SOURCE_DIR / "predist_dataset.zip"

BASE_PREDICTIONS_PATH = OUTPUT_DIR / "m1_event_type_one_vs_normal_predictions.csv"
BASE_DATASET_SUMMARY_PATH = OUTPUT_DIR / "m1_event_type_one_vs_normal_dataset_summary.csv"
BASE_SPECIAL_EVENT_AUDIT_PATH = OUTPUT_DIR / "m1_event_type_one_vs_normal_special_event_audit.csv"
FEATURE_SET_PATH = OUTPUT_DIR / "m1_compact_feature_set_summary.csv"

OUT_DATASET_SUMMARY = OUTPUT_DIR / "m1_class_binary_boosting_dataset_summary.csv"
OUT_CV_METRICS = OUTPUT_DIR / "m1_class_binary_boosting_cv_metrics.csv"
OUT_PREDICTIONS = OUTPUT_DIR / "m1_class_binary_boosting_predictions.csv"
OUT_THRESHOLD_METRICS = OUTPUT_DIR / "m1_class_binary_boosting_threshold_metrics.csv"
OUT_CLASS_METRICS = OUTPUT_DIR / "m1_class_binary_boosting_class_metrics.csv"
OUT_CONFUSION = OUTPUT_DIR / "m1_class_binary_boosting_confusion_matrix.csv"
OUT_FEATURE_IMPORTANCE = OUTPUT_DIR / "m1_class_binary_boosting_feature_importance.csv"
OUT_FOLD_STABILITY = OUTPUT_DIR / "m1_class_binary_boosting_fold_stability.csv"
OUT_DECISION_MATRIX = OUTPUT_DIR / "m1_class_binary_boosting_decision_matrix.csv"
OUT_QUALITY_AUDIT = OUTPUT_DIR / "m1_class_binary_boosting_quality_audit.csv"
OUT_REPORT = OUTPUT_DIR / "20_M1_class_binary_boosting_gate_validation_보고서.md"

RANDOM_STATE = 42
N_SPLITS = 5
THRESHOLDS = [0.4, 0.5, 0.6]
DEFAULT_THRESHOLD = 0.5
TARGET_CLASSES = ["fault", "task", "activity"]
SELECTED_DATASETS = {
    "fault": ["fault_no_overlap"],
    "task": ["task_pre_all", "task_post_all", "task_no_overlap"],
    "activity": ["activity_pre_all", "activity_post_all", "activity_no_overlap"],
}

for path in [BASE_PREDICTIONS_PATH, BASE_DATASET_SUMMARY_PATH, BASE_SPECIAL_EVENT_AUDIT_PATH, FEATURE_SET_PATH]:
    assert path.exists(), path

print("project_root:", PROJECT_ROOT)
print("output_dir:", OUTPUT_DIR)


project_root: C:\3rd_Project\HeatGridAgent
output_dir: C:\3rd_Project\HeatGridAgent\07_데이터산출물


## Data

In [2]:

base_predictions = pd.read_csv(BASE_PREDICTIONS_PATH)
base_dataset_summary = pd.read_csv(BASE_DATASET_SUMMARY_PATH)
special_event_audit = pd.read_csv(BASE_SPECIAL_EVENT_AUDIT_PATH)
feature_sets = pd.read_csv(FEATURE_SET_PATH)

print("base_predictions", base_predictions.shape)
print("base_dataset_summary", base_dataset_summary.shape)
print("special_event_audit", special_event_audit.shape)
print(feature_sets[["feature_set", "feature_count"]].to_string(index=False))


base_predictions (5208, 24)
base_dataset_summary (8, 12)
special_event_audit (8, 7)
      feature_set  feature_count
           base70             70
      expanded154            154
compact13_overlap             13
   compact20_main             20
  compact27_union             27


In [3]:

METADATA_COLS = {
    "source_id", "final_class", "target_class", "policy_variant", "source_event_id", "fault_event_id",
    "disturbance_row_id", "substation_id", "window_policy", "window_start", "window_end", "coverage_rate",
    "overlap_disturbance_count", "hard_normal_tag", "dataset", "feature_set", "feature_count", "model",
    "fold", "y_true", "probability_target", "prediction_t0_4", "prediction_t0_5", "prediction_t0_6",
}


def parse_feature_list(feature_set_name: str) -> list[str]:
    row = feature_sets.loc[feature_sets["feature_set"].eq(feature_set_name)]
    assert len(row) == 1, feature_set_name
    return [feature for feature in str(row.iloc[0]["features"]).split("|") if feature]


FEATURE_COLUMNS = {
    "compact13": parse_feature_list("compact13_overlap"),
    "expanded154": parse_feature_list("expanded154"),
}
assert len(FEATURE_COLUMNS["compact13"]) == 13
assert len(FEATURE_COLUMNS["expanded154"]) == 154

# 19번 prediction은 feature 값을 저장하지 않으므로, 16번 window audit에서 feature 포함 dataset을 재구성한다.
window_audit = pd.read_csv(OUTPUT_DIR / "m1_4class_window_policy_audit.csv")


def as_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.lower().isin(["true", "1", "yes"])


def overlap_count(frame: pd.DataFrame) -> pd.Series:
    return pd.to_numeric(frame["overlap_disturbance_count"], errors="coerce").fillna(0)


def normal_rows() -> pd.DataFrame:
    normal = window_audit.loc[
        window_audit["final_class"].eq("normal")
        & window_audit["window_policy"].eq("normal_event_7d")
        & as_bool(window_audit["coverage_eligible"])
    ].copy()
    assert len(normal) == 35
    return normal


def pick_event_rows(class_name: str, policy_variant: str) -> pd.DataFrame:
    eligible = as_bool(window_audit["coverage_eligible"])
    base = window_audit.loc[window_audit["final_class"].eq(class_name) & eligible].copy()
    if class_name == "fault":
        policies = ["fault_pre_7d"]
        require_no_overlap = policy_variant == "no_overlap"
    elif class_name == "task":
        if policy_variant == "pre_all":
            policies = ["task_pre_7d"]; require_no_overlap = False
        elif policy_variant == "post_all":
            policies = ["task_post_7d"]; require_no_overlap = False
        elif policy_variant == "no_overlap":
            policies = ["task_post_7d", "task_pre_7d"]; require_no_overlap = True
        else:
            raise ValueError(policy_variant)
    elif class_name == "activity":
        if policy_variant == "pre_all":
            policies = ["activity_pre_7d"]; require_no_overlap = False
        elif policy_variant == "post_all":
            policies = ["activity_post_7d"]; require_no_overlap = False
        elif policy_variant == "no_overlap":
            policies = ["activity_pre_7d", "activity_post_7d"]; require_no_overlap = True
        else:
            raise ValueError(policy_variant)
    else:
        raise ValueError(class_name)
    if require_no_overlap:
        base = base.loc[overlap_count(base).eq(0)].copy()
    rank = {policy: i for i, policy in enumerate(policies)}
    base["policy_rank"] = base["window_policy"].map(rank)
    base = base.loc[base["policy_rank"].notna()].copy()
    base = base.sort_values(["disturbance_row_id", "policy_rank", "coverage_rate"], ascending=[True, True, False])
    return base.drop_duplicates(subset=["disturbance_row_id"], keep="first").drop(columns=["policy_rank"])


def finalize_dataset(target_class: str, policy_variant: str) -> pd.DataFrame:
    event_rows = pick_event_rows(target_class, policy_variant)
    data = pd.concat([normal_rows(), event_rows], ignore_index=True).copy()
    data["target_class"] = target_class
    data["policy_variant"] = policy_variant
    data["dataset"] = f"{target_class}_{policy_variant}"
    data["binary_label"] = np.where(data["final_class"].eq(target_class), 1, 0)
    data["binary_label_name"] = np.where(data["binary_label"].eq(1), target_class, "normal")
    data["substation_id"] = data["substation_id"].astype(int)
    assert data["binary_label"].nunique() == 2
    assert not data["source_id"].duplicated().any()
    return data


DATASETS = {}
for target_class, dataset_names in SELECTED_DATASETS.items():
    for dataset_name in dataset_names:
        variant = dataset_name.replace(f"{target_class}_", "")
        DATASETS[dataset_name] = finalize_dataset(target_class, variant)

for name, data in DATASETS.items():
    print(name, data.shape, data["binary_label"].value_counts().to_dict())


fault_no_overlap (90, 195) {1: 55, 0: 35}
task_pre_all (72, 195) {1: 37, 0: 35}
task_post_all (76, 195) {1: 41, 0: 35}
task_no_overlap (72, 195) {1: 37, 0: 35}
activity_pre_all (82, 195) {1: 47, 0: 35}
activity_post_all (81, 195) {1: 46, 0: 35}
activity_no_overlap (81, 195) {1: 46, 0: 35}


In [4]:

summary_rows = []
for dataset_name, data in DATASETS.items():
    target = data.loc[data["binary_label"].eq(1)]
    summary_rows.append({
        "dataset": dataset_name,
        "target_class": data["target_class"].iloc[0],
        "policy_variant": data["policy_variant"].iloc[0],
        "rows": len(data),
        "normal_rows": int((data["binary_label"] == 0).sum()),
        "target_rows": int((data["binary_label"] == 1).sum()),
        "substation_count": int(data["substation_id"].nunique()),
        "target_substation_count": int(target["substation_id"].nunique()),
        "target_overlap_rows": int((overlap_count(target) > 0).sum()),
        "coverage_min": float(data["coverage_rate"].min()),
        "coverage_median": float(data["coverage_rate"].median()),
        "window_policies": "|".join(sorted(target["window_policy"].dropna().unique())),
    })
dataset_summary = pd.DataFrame(summary_rows)
dataset_summary.to_csv(OUT_DATASET_SUMMARY, index=False, encoding="utf-8-sig")
dataset_summary


,dataset,target_class,policy_variant,rows,normal_rows,target_rows,substation_count,target_substation_count,target_overlap_rows,coverage_min,coverage_median,window_policies
0,fault_no_overlap,fault,no_overlap,90,35,55,31,26,0,0.995040,1.0,fault_pre_7d
1,task_pre_all,task,pre_all,72,35,37,30,21,27,1.000000,1.0,task_pre_7d
2,task_post_all,task,post_all,76,35,41,31,22,4,0.982143,1.0,task_post_7d
3,task_no_overlap,task,no_overlap,72,35,37,31,22,0,0.982143,1.0,task_post_7d
4,activity_pre_all,activity,pre_all,82,35,47,27,19,4,0.995040,1.0,activity_pre_7d
5,activity_post_all,activity,post_all,81,35,46,27,19,5,0.995040,1.0,activity_post_7d
6,activity_no_overlap,activity,no_overlap,81,35,46,27,19,0,0.995040,1.0,activity_post_7d|activity_pre_7d


## Models

In [5]:

MODEL_NAMES = ["dummy_most_frequent", "logistic_balanced", "lightgbm_balanced", "xgboost_balanced"]


def build_model(model_name: str, y_train: np.ndarray | None = None):
    if model_name == "dummy_most_frequent":
        return DummyClassifier(strategy="most_frequent")
    if model_name == "logistic_balanced":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=5000, solver="liblinear", random_state=RANDOM_STATE)),
        ])
    if model_name == "lightgbm_balanced":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("model", LGBMClassifier(
                n_estimators=120,
                learning_rate=0.03,
                max_depth=2,
                num_leaves=4,
                min_child_samples=5,
                subsample=0.9,
                colsample_bytree=0.9,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                verbose=-1,
            )),
        ])
    if model_name == "xgboost_balanced":
        y = np.asarray(y_train) if y_train is not None else np.array([0, 1])
        neg = max(int((y == 0).sum()), 1)
        pos = max(int((y == 1).sum()), 1)
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("model", XGBClassifier(
                n_estimators=120,
                learning_rate=0.03,
                max_depth=2,
                min_child_weight=2,
                subsample=0.9,
                colsample_bytree=0.9,
                reg_lambda=2.0,
                objective="binary:logistic",
                eval_metric="logloss",
                scale_pos_weight=neg / pos,
                random_state=RANDOM_STATE,
                n_jobs=1,
            )),
        ])
    raise ValueError(model_name)


def clean_feature_matrix(data: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    X = data[feature_cols].apply(pd.to_numeric, errors="coerce")
    return X.replace([np.inf, -np.inf], np.nan)


def get_probability(model, X: pd.DataFrame) -> np.ndarray:
    proba = model.predict_proba(X)
    classes = list(model.classes_) if hasattr(model, "classes_") else list(model[-1].classes_)
    return proba[:, classes.index(1)]


def metric_row(y_true, y_prob, threshold: float) -> dict:
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = np.nan
    return {
        "threshold": threshold,
        "rows": int(len(y_true)),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "roc_auc": auc,
        "precision_target": precision_score(y_true, y_pred, zero_division=0),
        "recall_target": recall_score(y_true, y_pred, zero_division=0),
        "f1_target": f1_score(y_true, y_pred, zero_division=0),
        "normal_fpr": fp / (fp + tn) if (fp + tn) else np.nan,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def extract_importance(model, feature_cols: list[str]) -> np.ndarray:
    estimator = model[-1] if isinstance(model, Pipeline) else model
    if hasattr(estimator, "coef_"):
        values = np.abs(estimator.coef_[0])
    elif hasattr(estimator, "feature_importances_"):
        values = np.asarray(estimator.feature_importances_, dtype=float)
    else:
        values = np.zeros(len(feature_cols))
    return values


## Evaluation

In [6]:

prediction_rows = []
fold_metric_rows = []
importance_rows = []
for dataset_name, data in DATASETS.items():
    y = data["binary_label"].astype(int).to_numpy()
    groups = data["substation_id"].astype(int).to_numpy()
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    splits = list(splitter.split(data, y, groups))
    for feature_set, feature_cols in FEATURE_COLUMNS.items():
        X = clean_feature_matrix(data, feature_cols)
        for fold, (train_idx, test_idx) in enumerate(splits, start=1):
            train_groups = set(groups[train_idx])
            test_groups = set(groups[test_idx])
            assert not (train_groups & test_groups), f"group overlap {dataset_name} {feature_set} fold {fold}"
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            test_data = data.iloc[test_idx].copy()
            for model_name in MODEL_NAMES:
                model = build_model(model_name, y_train)
                model.fit(X_train, y_train)
                y_prob = get_probability(model, X_test)
                pred_frame = test_data[[
                    "source_id", "final_class", "target_class", "policy_variant", "source_event_id", "fault_event_id",
                    "disturbance_row_id", "substation_id", "window_policy", "window_start", "window_end", "coverage_rate",
                    "overlap_disturbance_count", "hard_normal_tag",
                ]].copy()
                pred_frame["dataset"] = dataset_name
                pred_frame["feature_set"] = feature_set
                pred_frame["feature_count"] = len(feature_cols)
                pred_frame["model"] = model_name
                pred_frame["fold"] = fold
                pred_frame["y_true"] = y_test
                pred_frame["probability_target"] = y_prob
                for threshold in THRESHOLDS:
                    pred_col = f"prediction_t{str(threshold).replace('.', '_')}"
                    pred_frame[pred_col] = (y_prob >= threshold).astype(int)
                    row = metric_row(y_test, y_prob, threshold)
                    row.update({
                        "dataset": dataset_name,
                        "target_class": data["target_class"].iloc[0],
                        "policy_variant": data["policy_variant"].iloc[0],
                        "feature_set": feature_set,
                        "feature_count": len(feature_cols),
                        "model": model_name,
                        "fold": fold,
                        "metric_scope": "fold",
                        "train_rows": int(len(train_idx)),
                        "test_rows": int(len(test_idx)),
                        "train_groups": "|".join(map(str, sorted(train_groups))),
                        "test_groups": "|".join(map(str, sorted(test_groups))),
                        "group_overlap_count": 0,
                    })
                    fold_metric_rows.append(row)
                prediction_rows.extend(pred_frame.to_dict("records"))
                for feature, value in zip(feature_cols, extract_importance(model, feature_cols)):
                    importance_rows.append({
                        "dataset": dataset_name,
                        "target_class": data["target_class"].iloc[0],
                        "policy_variant": data["policy_variant"].iloc[0],
                        "feature_set": feature_set,
                        "model": model_name,
                        "fold": fold,
                        "feature": feature,
                        "importance": float(value),
                    })

predictions = pd.DataFrame(prediction_rows)
fold_metrics = pd.DataFrame(fold_metric_rows)
feature_importance_fold = pd.DataFrame(importance_rows)
predictions.to_csv(OUT_PREDICTIONS, index=False, encoding="utf-8-sig")
print("predictions", predictions.shape)
print("fold_metrics", fold_metrics.shape)
print("feature_importance_fold", feature_importance_fold.shape)


predictions (4432, 24)
fold_metrics (840, 27)
feature_importance_fold (23380, 8)


In [7]:

aggregate_rows = []
class_rows = []
confusion_rows = []
for keys, group in predictions.groupby(["dataset", "target_class", "policy_variant", "feature_set", "feature_count", "model"], dropna=False):
    dataset, target_class, policy_variant, feature_set, feature_count, model_name = keys
    y_true = group["y_true"].astype(int).to_numpy()
    y_prob = group["probability_target"].astype(float).to_numpy()
    for threshold in THRESHOLDS:
        pred_col = f"prediction_t{str(threshold).replace('.', '_')}"
        y_pred = group[pred_col].astype(int).to_numpy()
        row = metric_row(y_true, y_prob, threshold)
        row.update({
            "dataset": dataset, "target_class": target_class, "policy_variant": policy_variant,
            "feature_set": feature_set, "feature_count": int(feature_count), "model": model_name,
            "fold": "all", "metric_scope": "aggregate", "train_rows": np.nan, "test_rows": int(len(group)),
            "train_groups": np.nan, "test_groups": np.nan, "group_overlap_count": 0,
        })
        aggregate_rows.append(row)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        confusion_rows.append({"dataset": dataset, "target_class": target_class, "policy_variant": policy_variant, "feature_set": feature_set, "model": model_name, "threshold": threshold, "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)})
        p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
        for label_value, label_name, precision, recall, f1, support in zip([0, 1], ["normal", str(target_class)], p, r, f, s):
            class_rows.append({"dataset": dataset, "target_class": target_class, "policy_variant": policy_variant, "feature_set": feature_set, "model": model_name, "threshold": threshold, "class_label": label_name, "class_value": label_value, "precision": precision, "recall": recall, "f1": f1, "support": int(support)})

aggregate_metrics = pd.DataFrame(aggregate_rows)
threshold_metrics = pd.concat([fold_metrics, aggregate_metrics], ignore_index=True)
cv_metrics = pd.concat([fold_metrics.loc[fold_metrics["threshold"].eq(DEFAULT_THRESHOLD)].copy(), aggregate_metrics.loc[aggregate_metrics["threshold"].eq(DEFAULT_THRESHOLD)].copy()], ignore_index=True)
class_metrics = pd.DataFrame(class_rows)
confusion_matrix_audit = pd.DataFrame(confusion_rows)

threshold_metrics.to_csv(OUT_THRESHOLD_METRICS, index=False, encoding="utf-8-sig")
cv_metrics.to_csv(OUT_CV_METRICS, index=False, encoding="utf-8-sig")
class_metrics.to_csv(OUT_CLASS_METRICS, index=False, encoding="utf-8-sig")
confusion_matrix_audit.to_csv(OUT_CONFUSION, index=False, encoding="utf-8-sig")
aggregate_metrics.sort_values(["threshold", "balanced_accuracy"], ascending=[True, False]).head(20)


,threshold,rows,accuracy,balanced_accuracy,macro_f1,roc_auc,precision_target,recall_target,f1_target,normal_fpr,...,feature_set,feature_count,model,fold,metric_scope,train_rows,test_rows,train_groups,test_groups,group_overlap_count
87,0.4,90,0.855556,0.819481,0.836112,0.895065,0.818182,0.981818,0.892562,0.342857,...,expanded154,154,lightgbm_balanced,all,aggregate,NaN,90,NaN,NaN,0
66,0.4,82,0.804878,0.800608,0.800608,0.885106,0.829787,0.829787,0.829787,0.228571,...,expanded154,154,logistic_balanced,all,aggregate,NaN,82,NaN,NaN,0
78,0.4,90,0.800000,0.784416,0.787290,0.845714,0.824561,0.854545,0.839286,0.285714,...,compact13,13,logistic_balanced,all,aggregate,NaN,90,NaN,NaN,0
93,0.4,90,0.822222,0.781818,0.796495,0.878961,0.791045,0.963636,0.868852,0.400000,...,expanded154,154,xgboost_balanced,all,aggregate,NaN,90,NaN,NaN,0
75,0.4,90,0.800000,0.774026,0.781789,0.841039,0.803279,0.890909,0.844828,0.342857,...,compact13,13,lightgbm_balanced,all,aggregate,NaN,90,NaN,NaN,0
81,0.4,90,0.788889,0.759740,0.768010,0.858701,0.790323,0.890909,0.837607,0.371429,...,compact13,13,xgboost_balanced,all,aggregate,NaN,90,NaN,NaN,0
18,0.4,81,0.728395,0.730124,0.726351,0.821739,0.785714,0.717391,0.750000,0.257143,...,expanded154,154,logistic_balanced,all,aggregate,NaN,81,NaN,NaN,0
150,0.4,72,0.722222,0.722780,0.722222,0.730502,0.742857,0.702703,0.722222,0.257143,...,compact13,13,logistic_balanced,all,aggregate,NaN,72,NaN,NaN,0
129,0.4,76,0.723684,0.720906,0.721320,0.694774,0.738095,0.756098,0.746988,0.314286,...,compact13,13,xgboost_balanced,all,aggregate,NaN,76,NaN,NaN,0
147,0.4,72,0.708333,0.706950,0.706920,0.760618,0.700000,0.756757,0.727273,0.342857,...,compact13,13,lightgbm_balanced,all,aggregate,NaN,72,NaN,NaN,0


## Feature Importance And Fold Stability

In [8]:

feature_importance = (
    feature_importance_fold
    .groupby(["dataset", "target_class", "policy_variant", "feature_set", "model", "feature"], as_index=False)
    .agg(importance_mean=("importance", "mean"), importance_std=("importance", "std"), nonzero_fold_count=("importance", lambda s: int((s > 0).sum())))
)
feature_importance.to_csv(OUT_FEATURE_IMPORTANCE, index=False, encoding="utf-8-sig")

fold_default = fold_metrics.loc[fold_metrics["threshold"].eq(DEFAULT_THRESHOLD)].copy()
fold_stability = (
    fold_default
    .groupby(["dataset", "target_class", "policy_variant", "feature_set", "feature_count", "model"], as_index=False)
    .agg(
        fold_balanced_accuracy_mean=("balanced_accuracy", "mean"),
        fold_balanced_accuracy_std=("balanced_accuracy", "std"),
        fold_macro_f1_mean=("macro_f1", "mean"),
        fold_macro_f1_std=("macro_f1", "std"),
        fold_recall_mean=("recall_target", "mean"),
        fold_recall_std=("recall_target", "std"),
        fold_normal_fpr_mean=("normal_fpr", "mean"),
        fold_normal_fpr_std=("normal_fpr", "std"),
        folds=("fold", "count"),
    )
)
fold_stability.to_csv(OUT_FOLD_STABILITY, index=False, encoding="utf-8-sig")
fold_stability.sort_values("fold_balanced_accuracy_mean", ascending=False).head(15)


,dataset,target_class,policy_variant,feature_set,feature_count,model,fold_balanced_accuracy_mean,fold_balanced_accuracy_std,fold_macro_f1_mean,fold_macro_f1_std,fold_recall_mean,fold_recall_std,fold_normal_fpr_mean,fold_normal_fpr_std,folds
31,fault_no_overlap,fault,no_overlap,expanded154,154,xgboost_balanced,0.815422,0.106352,0.825039,0.110393,0.927273,0.099586,0.296429,0.133392,5
29,fault_no_overlap,fault,no_overlap,expanded154,154,lightgbm_balanced,0.800649,0.092734,0.801663,0.086967,0.872727,0.103652,0.271429,0.204540,5
27,fault_no_overlap,fault,no_overlap,compact13,13,xgboost_balanced,0.795455,0.140100,0.797914,0.138647,0.890909,0.076060,0.300000,0.239046,5
22,activity_pre_all,activity,pre_all,expanded154,154,logistic_balanced,0.786786,0.010767,0.785682,0.010632,0.779524,0.084944,0.205952,0.096311,5
25,fault_no_overlap,fault,no_overlap,compact13,13,lightgbm_balanced,0.784848,0.134718,0.781955,0.133219,0.836364,0.076060,0.266667,0.214550,5
26,fault_no_overlap,fault,no_overlap,compact13,13,logistic_balanced,0.741180,0.101518,0.735239,0.117529,0.745455,0.197086,0.263095,0.079904,5
50,task_pre_all,task,pre_all,compact13,13,logistic_balanced,0.737698,0.051327,0.730133,0.046169,0.675397,0.145976,0.200000,0.191663,5
15,activity_post_all,activity,post_all,expanded154,154,xgboost_balanced,0.735476,0.099561,0.723986,0.102302,0.748333,0.158645,0.277381,0.267778,5
6,activity_no_overlap,activity,no_overlap,expanded154,154,logistic_balanced,0.733333,0.103390,0.726166,0.104605,0.692857,0.123832,0.226190,0.212959,5
1,activity_no_overlap,activity,no_overlap,compact13,13,lightgbm_balanced,0.718611,0.063922,0.717520,0.063450,0.758651,0.096107,0.321429,0.101015,5


## Decision Matrix

In [9]:

dummy_lookup = aggregate_metrics.loc[aggregate_metrics["model"].eq("dummy_most_frequent"), ["dataset", "feature_set", "threshold", "balanced_accuracy"]].rename(columns={"balanced_accuracy": "dummy_balanced_accuracy"})
decision_matrix = aggregate_metrics.merge(dummy_lookup, on=["dataset", "feature_set", "threshold"], how="left")
decision_matrix["balanced_accuracy_lift_vs_dummy"] = decision_matrix["balanced_accuracy"] - decision_matrix["dummy_balanced_accuracy"]
stability_cols = ["dataset", "feature_set", "model", "fold_balanced_accuracy_std", "fold_recall_std", "fold_normal_fpr_std"]
decision_matrix = decision_matrix.merge(fold_stability[stability_cols], on=["dataset", "feature_set", "model"], how="left")

logistic_ref = decision_matrix.loc[
    decision_matrix["model"].eq("logistic_balanced") & decision_matrix["threshold"].eq(DEFAULT_THRESHOLD),
    ["target_class", "dataset", "feature_set", "balanced_accuracy", "recall_target", "normal_fpr", "fold_balanced_accuracy_std"],
].rename(columns={
    "balanced_accuracy": "logistic_balanced_accuracy",
    "recall_target": "logistic_recall_target",
    "normal_fpr": "logistic_normal_fpr",
    "fold_balanced_accuracy_std": "logistic_fold_ba_std",
})
decision_matrix = decision_matrix.merge(logistic_ref, on=["target_class", "dataset", "feature_set"], how="left")
decision_matrix["delta_ba_vs_logistic"] = decision_matrix["balanced_accuracy"] - decision_matrix["logistic_balanced_accuracy"]
decision_matrix["delta_recall_vs_logistic"] = decision_matrix["recall_target"] - decision_matrix["logistic_recall_target"]
decision_matrix["delta_fpr_vs_logistic"] = decision_matrix["normal_fpr"] - decision_matrix["logistic_normal_fpr"]
decision_matrix["delta_fold_ba_std_vs_logistic"] = decision_matrix["fold_balanced_accuracy_std"] - decision_matrix["logistic_fold_ba_std"]

decision_matrix["passes_gate_base"] = (
    (decision_matrix["recall_target"] >= 0.80)
    & (decision_matrix["normal_fpr"] <= 0.25)
    & (decision_matrix["balanced_accuracy_lift_vs_dummy"] >= 0.15)
    & decision_matrix["group_overlap_count"].fillna(0).eq(0)
    & ~decision_matrix["model"].eq("dummy_most_frequent")
)
decision_matrix["passes_boosting_vs_logistic"] = (
    decision_matrix["model"].isin(["lightgbm_balanced", "xgboost_balanced"])
    & (decision_matrix["delta_ba_vs_logistic"] >= 0.03)
    & (decision_matrix["delta_recall_vs_logistic"] >= -0.05)
    & (decision_matrix["delta_fpr_vs_logistic"] <= 0.05)
    & (decision_matrix["delta_fold_ba_std_vs_logistic"].fillna(0) <= 0)
)

def row_decision(row: pd.Series) -> str:
    if row["passes_gate_base"] and row["passes_boosting_vs_logistic"]:
        return "boosting_gate_candidate"
    if row["passes_gate_base"] and row["model"] == "logistic_balanced":
        return "logistic_gate_candidate"
    if row["passes_gate_base"]:
        return "logistic_or_existing_tree_candidate"
    if row["balanced_accuracy_lift_vs_dummy"] >= 0.15:
        return "gate_iteration_needed"
    return "window_label_review_required"

decision_matrix["decision_candidate"] = decision_matrix.apply(row_decision, axis=1)

target_rows = []
for target, group in decision_matrix.loc[decision_matrix["threshold"].eq(DEFAULT_THRESHOLD) & ~decision_matrix["model"].eq("dummy_most_frequent")].groupby("target_class"):
    if (group["decision_candidate"] == "boosting_gate_candidate").any():
        target_decision = "boosting_gate_candidate"
    elif (group["decision_candidate"] == "logistic_gate_candidate").any():
        target_decision = "logistic_gate_candidate"
    elif group["passes_gate_base"].any():
        target_decision = "logistic_or_existing_tree_candidate"
    elif (group["balanced_accuracy_lift_vs_dummy"] >= 0.15).any():
        target_decision = "gate_iteration_needed"
    else:
        target_decision = "window_label_review_required"
    target_rows.append({"target_class": target, "target_decision": target_decision})
target_decisions = pd.DataFrame(target_rows)
decision_matrix = decision_matrix.merge(target_decisions, on="target_class", how="left")

stable_targets = target_decisions.loc[target_decisions["target_decision"].isin(["boosting_gate_candidate", "logistic_gate_candidate", "logistic_or_existing_tree_candidate"]), "target_class"].tolist()
if stable_targets == ["fault"] or (len(stable_targets) == 1 and "fault" in stable_targets):
    overall_decision = "fault_gate_first_strategy"
elif len(stable_targets) >= 2:
    overall_decision = "partial_multiclass_ready"
else:
    overall_decision = "return_to_window_label_design"
decision_matrix["overall_decision"] = overall_decision
decision_matrix.to_csv(OUT_DECISION_MATRIX, index=False, encoding="utf-8-sig")
print("overall_decision", overall_decision)
decision_matrix.loc[decision_matrix["threshold"].eq(DEFAULT_THRESHOLD) & ~decision_matrix["model"].eq("dummy_most_frequent")].sort_values(["passes_boosting_vs_logistic", "passes_gate_base", "balanced_accuracy"], ascending=[False, False, False]).head(20)


overall_decision return_to_window_label_design


,threshold,rows,accuracy,balanced_accuracy,macro_f1,roc_auc,precision_target,recall_target,f1_target,normal_fpr,...,logistic_fold_ba_std,delta_ba_vs_logistic,delta_recall_vs_logistic,delta_fpr_vs_logistic,delta_fold_ba_std_vs_logistic,passes_gate_base,passes_boosting_vs_logistic,decision_candidate,target_decision,overall_decision
46,0.5,81,0.740741,0.740994,0.738187,0.761491,0.790698,0.739130,0.764045,0.257143,...,0.127608,0.082298,0.021739,-0.142857,-0.028048,False,True,gate_iteration_needed,gate_iteration_needed,return_to_window_label_design
4,0.5,81,0.728395,0.723292,0.723292,0.793168,0.760870,0.760870,0.760870,0.314286,...,0.101849,0.187578,0.260870,-0.114286,-0.037927,False,True,gate_iteration_needed,gate_iteration_needed,return_to_window_label_design
10,0.5,81,0.703704,0.704969,0.701474,0.786957,0.761905,0.695652,0.727273,0.285714,...,0.101849,0.169255,0.195652,-0.142857,-0.024599,False,True,gate_iteration_needed,gate_iteration_needed,return_to_window_label_design
136,0.5,76,0.684211,0.686411,0.683992,0.721951,0.729730,0.658537,0.692308,0.285714,...,0.083895,0.111847,0.195122,-0.028571,-0.004913,False,True,gate_iteration_needed,gate_iteration_needed,return_to_window_label_design
160,0.5,72,0.666667,0.669498,0.664336,0.745946,0.724138,0.567568,0.636364,0.228571,...,0.118608,0.195367,0.162162,-0.228571,-0.034899,False,True,gate_iteration_needed,gate_iteration_needed,return_to_window_label_design
166,0.5,72,0.638889,0.640154,0.638610,0.755212,0.666667,0.594595,0.628571,0.314286,...,0.118608,0.166023,0.189189,-0.142857,-0.068657,False,True,gate_iteration_needed,gate_iteration_needed,return_to_window_label_design
94,0.5,90,0.844444,0.820779,0.830280,0.878961,0.836066,0.927273,0.879310,0.285714,...,0.084378,0.114286,0.200000,-0.028571,0.021974,False,False,gate_iteration_needed,gate_iteration_needed,return_to_window_label_design
88,0.5,90,0.822222,0.807792,0.810924,0.895065,0.842105,0.872727,0.857143,0.257143,...,0.084378,0.101299,0.145455,-0.057143,0.008356,False,False,gate_iteration_needed,gate_iteration_needed,return_to_window_label_design
82,0.5,90,0.822222,0.802597,0.808612,0.858701,0.830508,0.890909,0.859649,0.285714,...,0.101518,0.058442,0.145455,0.028571,0.038583,False,False,gate_iteration_needed,gate_iteration_needed,return_to_window_label_design
67,0.5,82,0.792683,0.793617,0.790155,0.885106,0.840909,0.787234,0.813187,0.200000,...,0.010767,0.000000,0.000000,0.000000,0.000000,False,False,gate_iteration_needed,gate_iteration_needed,return_to_window_label_design


## Quality Audit And Report

In [10]:

quality_rows = []
recheck_predictions = pd.read_csv(OUT_PREDICTIONS)
recheck_decision = pd.read_csv(OUT_DECISION_MATRIX)
for _, row in recheck_decision.loc[recheck_decision["metric_scope"].eq("aggregate")].iterrows():
    pred_col = f"prediction_t{str(row['threshold']).replace('.', '_')}"
    subset = recheck_predictions.loc[
        recheck_predictions["dataset"].eq(row["dataset"]) &
        recheck_predictions["feature_set"].eq(row["feature_set"]) &
        recheck_predictions["model"].eq(row["model"])
    ]
    y_true = subset["y_true"].astype(int).to_numpy()
    y_pred = subset[pred_col].astype(int).to_numpy()
    ba = balanced_accuracy_score(y_true, y_pred)
    macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    quality_rows.append({
        "check_name": "metric_recompute",
        "dataset": row["dataset"], "feature_set": row["feature_set"], "model": row["model"], "threshold": row["threshold"],
        "reported_balanced_accuracy": row["balanced_accuracy"], "recomputed_balanced_accuracy": ba,
        "reported_macro_f1": row["macro_f1"], "recomputed_macro_f1": macro,
        "pass": bool(np.isclose(ba, row["balanced_accuracy"]) and np.isclose(macro, row["macro_f1"])),
        "note": "independent recompute from saved predictions",
    })
source_status = subprocess.run(["git", "status", "--short", "--", "05_데이터셋/PreDist"], capture_output=True, text=True, encoding="utf-8", errors="ignore").stdout.strip()
quality_rows.append({"check_name": "source_zip_exists", "dataset": "source", "feature_set": "NA", "model": "NA", "threshold": np.nan, "reported_balanced_accuracy": np.nan, "recomputed_balanced_accuracy": np.nan, "reported_macro_f1": np.nan, "recomputed_macro_f1": np.nan, "pass": ZIP_PATH.exists(), "note": "original zip exists"})
quality_rows.append({"check_name": "source_metadata_clean", "dataset": "source", "feature_set": "NA", "model": "NA", "threshold": np.nan, "reported_balanced_accuracy": np.nan, "recomputed_balanced_accuracy": np.nan, "reported_macro_f1": np.nan, "recomputed_macro_f1": np.nan, "pass": source_status == "", "note": "source data directory has no git changes"})
quality_rows.append({"check_name": "special_event_policy_retained", "dataset": "audit", "feature_set": "NA", "model": "NA", "threshold": np.nan, "reported_balanced_accuracy": np.nan, "recomputed_balanced_accuracy": np.nan, "reported_macro_f1": np.nan, "recomputed_macro_f1": np.nan, "pass": set(special_event_audit["event_id"].astype(int)) == {20,34,69,67,19,68,35,48}, "note": "special events retained from 19 audit"})
quality_audit = pd.DataFrame(quality_rows)
quality_audit.to_csv(OUT_QUALITY_AUDIT, index=False, encoding="utf-8-sig")

required = [OUT_DATASET_SUMMARY, OUT_CV_METRICS, OUT_PREDICTIONS, OUT_THRESHOLD_METRICS, OUT_CLASS_METRICS, OUT_CONFUSION, OUT_FEATURE_IMPORTANCE, OUT_FOLD_STABILITY, OUT_DECISION_MATRIX, OUT_QUALITY_AUDIT]
for path in required:
    assert path.exists(), path
assert quality_audit["pass"].all()
assert threshold_metrics["group_overlap_count"].fillna(0).max() == 0
for path in required:
    text = path.read_text(encoding="utf-8-sig", errors="ignore").lower()
    assert ("manufacturer" + "_2") not in text
    assert ("manufacturer" + " " + "2") not in text

best_default = decision_matrix.loc[decision_matrix["threshold"].eq(DEFAULT_THRESHOLD) & ~decision_matrix["model"].eq("dummy_most_frequent")].sort_values(["passes_boosting_vs_logistic", "passes_gate_base", "balanced_accuracy"], ascending=[False, False, False])
best_by_target = best_default.groupby("target_class").head(1).copy()

def md_table(df: pd.DataFrame) -> str:
    table = df.copy()
    for col in table.columns:
        table[col] = table[col].map(lambda x: "" if pd.isna(x) else str(x))
    return "\n".join(["| " + " | ".join(table.columns) + " |", "| " + " | ".join(["---"] * len(table.columns)) + " |"] + ["| " + " | ".join(row) + " |" for row in table.astype(str).values.tolist()])

target_decision_table = target_decisions.copy()
show_cols = ["target_class", "dataset", "feature_set", "model", "balanced_accuracy", "macro_f1", "recall_target", "normal_fpr", "fold_balanced_accuracy_std", "delta_ba_vs_logistic", "delta_recall_vs_logistic", "delta_fpr_vs_logistic", "passes_boosting_vs_logistic", "decision_candidate", "target_decision"]
class_show = class_metrics.loc[class_metrics["threshold"].eq(DEFAULT_THRESHOLD)].merge(best_by_target[["target_class", "dataset", "feature_set", "model"]], on=["target_class", "dataset", "feature_set", "model"], how="inner")
conf_show = confusion_matrix_audit.loc[confusion_matrix_audit["threshold"].eq(DEFAULT_THRESHOLD)].merge(best_by_target[["target_class", "dataset", "feature_set", "model"]], on=["target_class", "dataset", "feature_set", "model"], how="inner")

lines = [
    "# M1 Class Binary Boosting Gate 검증 보고서",
    "",
    "## 결론",
    f"- 최종 판단: `{overall_decision}`",
    "- class별 binary gate에서 Logistic 기준선 대비 LightGBM/XGBoost를 비교했다.",
    "- boosting은 class별로만 채택하며, 모든 class에 일괄 적용하지 않는다.",
    "",
    "## 근거",
    md_table(best_by_target[show_cols]),
    "",
    "## Class별 의사결정",
    md_table(target_decision_table),
    "",
    "## Class별 Sample 수",
    md_table(dataset_summary),
    "",
    "## Class별 Precision/Recall/F1",
    md_table(class_show[["target_class", "dataset", "feature_set", "model", "class_label", "precision", "recall", "f1", "support"]]),
    "",
    "## Confusion Matrix",
    md_table(conf_show[["target_class", "dataset", "feature_set", "model", "tn", "fp", "fn", "tp"]]),
    "",
    "## Fold 안정성",
    md_table(fold_stability.merge(best_by_target[["target_class", "dataset", "feature_set", "model"]], on=["target_class", "dataset", "feature_set", "model"], how="inner")[["target_class", "dataset", "feature_set", "model", "fold_balanced_accuracy_mean", "fold_balanced_accuracy_std", "fold_recall_std", "fold_normal_fpr_std"]]),
    "",
    "## 품질 검증",
    "- 생성 CSV를 다시 읽어 balanced accuracy와 macro F1을 독립 재계산했고 저장 수치와 대조했다.",
    "- M1 산출물만 사용했고 비대상 제조사 문자열/경로/계산 결과는 새 산출물에 포함하지 않았다.",
    "- 원본 ZIP 존재와 metadata/source directory clean 상태를 확인했다.",
    "- Event 20/34/69/67, Event 19/68/35/48 처리 기준은 19번 audit에서 유지된 것을 확인했다.",
    md_table(quality_audit.groupby("check_name").agg(pass_all=("pass", "all"), rows=("check_name", "count")).reset_index()),
    "",
    "## 한계",
    "- normal 35건 고정이라 normal FPR이 fold 구성에 민감하다.",
    "- boosting은 작은 데이터에서 과적합 위험이 있어 fold 안정성을 반드시 같이 봐야 한다.",
    "- 이번 결과는 class별 gate 비교이며 최종 4분류 성능이 아니다.",
    "",
    "## 다음 작업 순서",
    "1. 안정적인 class gate부터 후보로 잠근다.",
    "2. boosting이 명확히 안정적인 class에만 boosting을 채택한다.",
    "3. 불안정한 task/activity는 window/label 정책 재검토 후 다시 비교한다.",
    "4. class별 gate가 2개 이상 안정화되면 4분류 전략을 다시 설계한다.",
]
OUT_REPORT.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(OUT_REPORT)


C:\3rd_Project\HeatGridAgent\07_데이터산출물\20_M1_class_binary_boosting_gate_validation_보고서.md


## Takeaways

보고서의 class별 decision을 기준으로 boosting 채택 여부와 다음 4분류 전략을 판단한다.